# A03: Multiprocessing y Multithreading

## Objetivos de aprendizaje

Al finalizar esta sesion, seras capaz de:

1. Explicar la diferencia fundamental entre **procesos** (memoria aislada y paralelismo real) e **hilos** (memoria compartida pero limitados por el GIL en Python).
2. Crear y gestionar **hilos** con `threading.Thread`, daemon threads, `join(timeout)` y `ThreadPoolExecutor` para tareas de E/S (I/O).
3. Sincronizar hilos con `Lock`, `RLock`, `Event` y `Condition`, diagnosticar **race conditions** y evitar **deadlocks** con timeouts.
4. Crear **procesos** con `multiprocessing.Process`, comunicarlos con `Queue` y `Pipe`, y escalar modelos CPU-bound con `ProcessPoolExecutor`, explicando `spawn` vs `fork`.
5. Compartir memoria de forma eficiente con `shared_memory.SharedMemory` y `Array`/`Value`, y dominar la **serializacion con pickle** (que se puede y que no se puede pasar a procesos).
6. Decidir con criterio entre hilos y procesos realizando un **benchmark objetivo** con `time.perf_counter` y presentar resultados en tabla.

## Analogia: Restaurantes y chefs

Para entender procesos e hilos, piensa en un **restaurante**.

**Los procesos son restaurantes completamente separados.** Cada restaurante tiene su propia cocina, su propio refrigerador y su propia memoria. Si un restaurante quema su cocina, los demas no se enteran. Pueden cocinar **al mismo tiempo** porque tienen espacios de trabajo independientes. Eso es paralelismo real.

**Los hilos son los chefs que trabajan en el MISMO restaurante.** Comparten la misma cocina, el mismo refrigerador y los mismos utensilios (la misma memoria). La ventaja es que pueden pasar ingredientes entre si sin copiarlos. La desventaja es que, en Python, hay una **llave unica (el GIL)** que solo permite a un chef usar la estufa a la vez.

```
+------------------------------------------------------------+
|  PROCESOS: restaurantes separados (memoria aislada)         |
|                                                            |
|  +----------------+    +----------------+                  |
|  | Restaurante A  |    | Restaurante B  |                  |
|  | cocina  | mem  |    | cocina  | mem  |    ... paralelo  |
|  +----------------+    +----------------+                  |
+------------------------------------------------------------+
|                                                            |
|  HILOS: chefs del mismo restaurante (memoria COMPARTIDA)   |
|                                                            |
|      +------------------------------------------------+    |
|      |  Cocina unica (memoria del proceso)             |    |
|      |  chef1 chef2 chef3 ...                          |    |
|      |    │    │    │                                    |    |
|      |    │    LLave unica = GIL (1 estufa a la vez)   |    |
|      +------------------------------------------------+    |
+------------------------------------------------------------+

  - Tarea de E/S (leer archivos, red, API): los chefs esperan,
    asi que la llave se libera -> los hilos ayudan.
  - Tarea de CPU (calcular, procesar pixeles): un chef bloquea
    la estufa -> los hilos NO ayudan, los PROCESOS si.
```

> **Regla de oro**: Hilos para esperar (I/O), procesos para calcular (CPU).

## 1. Threading avanzado

Un **hilo** (thread) es la unidad mas pequena de ejecucion dentro de un proceso. En Python todos los hilos viven en el mismo proceso, comparten la misma memoria y se coordinan mediante el **GIL** (Global Interpreter Lock).

API principal de `threading.Thread`:

- `threading.Thread(target=func, args=..., kwargs=..., daemon=...)` — crea un hilo.
- `run()` — metodo a sobreescribir con la logica del hilo.
- `start()` — inicia la ejecucion (no confundir con `run()`).
- `join(timeout)` — bloquea al hilo principal hasta que termine, con tiempo maximo opcional.
- `is_alive()` — indica si el hilo sigue ejecutandose.
- `daemon` — hilo que muere cuando el principal termina.

### 1.1 Crear un hilo con `target`, `args` y `kwargs`

In [ ]:
import threading
import time

def descargar(url: str, intentos: int = 3) -> None:
    nombre = threading.current_thread().name
    for i in range(intentos):
        time.sleep(0.3)
        print(f"[{nombre}] {url}: intento {i+1}/{intentos}")

# target, args (posicionales) y kwargs (nominales)
t = threading.Thread(target=descargar, args=("api/datos",), kwargs={"intentos": 2})
print("Antes de start:", t.is_alive())
t.start()
print("Despues de start:", t.is_alive())
t.join()
print("Despues de join:", t.is_alive())

### 1.2 Subclase de `Thread` sobreescribiendo `run()`

In [ ]:
class Procesador(threading.Thread):
    """Subclase de Thread: la logica va en run()."""
    def __init__(self, datos: list[int]):
        super().__init__()
        self.datos = datos

    def run(self) -> None:
        resultado = sum(self.datos)
        print(f"[{self.name}] suma = {resultado}")

hilo = Procesador([10, 20, 30, 40])
hilo.start()
hilo.join()
print("Tarea principal continuo.")

### 1.3 Daemon threads

Un **daemon thread** se ejecuta en segundo plano y **no impide que el programa termine**. Si el hilo principal termina, los daemons mueren abruptamente. Son utiles para tareas de monitoreo, log o limpieza. Un hilo **no-daemon** (por defecto) obliga al programa a esperar a que termine.

In [ ]:
def monitor() -> None:
    for i in range(10):
        time.sleep(0.2)
        print(f"Monitor: tick {i}...")

d = threading.Thread(target=monitor, daemon=True, name="monitor-daemon")
d.start()

time.sleep(0.5)
print("Main termina: el daemon muere con el, sin completar sus 10 ticks.")

### 1.4 `ThreadPoolExecutor` — multiples tareas I/O

`concurrent.futures.ThreadPoolExecutor` crea un grupo de hilos reutilizables. La API `submit()` devuelve un **Future** y `map()` aplica una funcion a muchos iterables en paralelo. Es la forma preferida de paralelizar tareas de **E/S**.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def peticion_api(id_: int) -> str:
    time.sleep(0.4)
    return f"respuesta_{id_}"

urls = list(range(8))

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(peticion_api, u): u for u in urls}
    for futuro in as_completed(futures):
        print("Lista:", futuro.result())

print("\nUso con map():")
with ThreadPoolExecutor(max_workers=4) as pool:
    for r in pool.map(peticion_api, range(4)):
        print(r)

### 1.5 Benchmark: hilos en tarea de E/S

En una tarea I/O-bound (esperar, no calcular), los hilos **si** aceleran porque mientras uno espera la E/S, otro avanza.

In [ ]:
def tarea_io(seg: float) -> None:
    time.sleep(seg)  # simula lectura de archivo/red

def secuencial(n: int) -> float:
    inicio = time.perf_counter()
    for _ in range(n):
        tarea_io(0.2)
    return time.perf_counter() - inicio

def con_hilos(n: int) -> float:
    inicio = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n) as pool:
        list(pool.map(tarea_io, [0.2] * n))
    return time.perf_counter() - inicio

N = 8
t_sec = secuencial(N)
t_hilos = con_hilos(N)
print(f"Secuencial: {t_sec:.2f}s")
print(f"Con hilos : {t_hilos:.2f}s")
print(f"Speedup   : {t_sec / t_hilos:.2f}x")

## 2. Sincronizacion con threads

Cuando varios hilos comparten memoria, el acceso concurrente a datos compartidos puede corromperlos. Se necesitan **mecanismos de sincronizacion**.

| Mecanismo | Uso |
|---|---|
| `Lock` | Zona critica: solo un hilo a la vez (exclusivo). |
| `RLock` | Lock reentrante: el mismo hilo puede adquirirlo varias veces. |
| `Event` | Señal de un hilo a otros (esperar/avisar). |
| `Condition` | Coordinacion mas rica: esperar condicion + notificar. |

### 2.1 Race condition (condicion de carrera)

Con `ThreadPoolExecutor`, dos o mas hilos incrementan una variable compartida. Sin sincronizacion, las operaciones `contador += 1` se intercalan y se pierden actualizaciones.

In [ ]:
# RACE CONDITION: contador compartido sin lock
contador = 0

def incrementar(veces: int) -> None:
    global contador
    for _ in range(veces):
        contador += 1

with ThreadPoolExecutor(max_workers=10) as pool:
    list(pool.map(incrementar, [100000] * 10))

print(f"Esperado: {100000 * 10}")
print(f"Obtenido: {contador}  <- probablemente diferente (race condition)")

### 2.2 Solucion: `Lock`

Un `Lock` garantiza que solo un hilo entre a la **seccion critica** a la vez (`with lock:`). Esto elimina la race condition a cambio de serializar esa parte.

In [ ]:
lock = threading.Lock()
contador_seguro = 0

def incrementar_seguro(veces: int) -> None:
    global contador_seguro
    for _ in range(veces):
        with lock:                 # seccion critica
            contador_seguro += 1

with ThreadPoolExecutor(max_workers=10) as pool:
    list(pool.map(incrementar_seguro, [100000] * 10))

print(f"Esperado: {100000 * 10}")
print(f"Obtenido: {contador_seguro}  <- correcto gracias al Lock")

### 2.3 `RLock` — lock reentrante

Un `Lock` normal **no es reentrante**: si el mismo hilo intenta adquirirlo dos veces genera un deadlock. `RLock` (Reentrant Lock) permite que el hilo que lo adquirio lo vuelva a adquirir, contando las adquisiciones.

In [ ]:
rlock = threading.RLock()

def funcion_externa() -> None:
    with rlock:
        print("externa: dentro del lock")
        funcion_interna()

def funcion_interna() -> None:
    with rlock:   # mismo hilo, RLock lo permite
        print("interna: re-entrando al mismo lock")

# Con Lock normal esto fallaria con deadlock; con RLock funciona
h = threading.Thread(target=funcion_externa)
h.start()
h.join()
print("RLock permitio la reentrada.")

### 2.4 `Event` — señalizacion entre hilos

Un `Event` es un flag que un hilo pone (`set()`) y otros esperan (`wait()`). Sirve para coordinar: un hilo productor avisa cuando los datos estan listos.

In [ ]:
evento = threading.Event()
datos_listos = []

def productor():
    print("[productor] generando datos...")
    time.sleep(0.5)
    datos_listos.extend([1, 2, 3, 4])
    evento.set()          # senal de que ya esta listo

def consumidor():
    print("[consumidor] esperando datos...")
    evento.wait()         # bloquea hasta que se haga set()
    print(f"[consumidor] datos recibidos: {datos_listos}")

c = threading.Thread(target=consumidor)
p = threading.Thread(target=productor)
c.start(); p.start()
c.join(); p.join()

### 2.5 `Condition` — coordinacion productor/consumidor

`Condition` combina un lock y la capacidad de **esperar** una condicion. Los hilos usan `wait()` (libera el lock y espera) y `notify()` (despierta a un hilo que espera). Es la base del patron productor-consumidor.

In [ ]:
cond = threading.Condition()
buffer = []
TOTAL = 5

def productor():
    global buffer
    for i in range(TOTAL):
        with cond:
            buffer.append(i)
            print(f"[prod] produjo {i}, buffer={buffer}")
            cond.notify()      # avisa a un consumidor
            time.sleep(0.1)

def consumidor():
    global buffer
    while True:
        with cond:
            while not buffer:   # espera mientras vacio
                cond.wait()
            item = buffer.pop(0)
            print(f"[cons] consumio {item}")
            if len(buffer) == 0 and item == TOTAL - 1:
                break

th = threading.Thread(target=consumidor, daemon=True)
th.start()
productor()
th.join(timeout=5)
print("Coordinacion productor/consumidor finalizada.")

### 2.6 Deadlock y como evitarlo

Un **deadlock** ocurre cuando dos hilos se quedan esperando el lock del otro para siempre. Estrategias para evitarlo:

1. **Orden fijo** de adquisicion de locks (todos en el mismo orden).
2. **Timeouts**: usar `lock.acquire(timeout=...)` y manejar el fracaso.
3. **Usar context managers** (`with lock:`) con bloques pequenos.

El ejemplo simula un deadlock y lo resuelve con `timeout`, siguiendo un orden fijo.

In [ ]:
A = threading.Lock()
B = threading.Lock()

def tarea_con_timeout(primero, segundo, nombre) -> None:
    if not primero.acquire(timeout=1):
        print(f"[{nombre}] no pudo adquirir el primer lock, aborta")
        return
    time.sleep(0.05)  # exagera la condicion de carrera
    if not segundo.acquire(timeout=1):
        print(f"[{nombre}] NO obtuvo el segundo lock -> timeout, libera")
        primero.release()
        return
    print(f"[{nombre}] obtuvo ambos locks, trabajo critico")
    segundo.release()
    primero.release()

# Ambos intentan en el MISMO orden (A -> B): sin deadlock
h1 = threading.Thread(target=tarea_con_timeout, args=(A, B, "h1"))
h2 = threading.Thread(target=tarea_con_timeout, args=(A, B, "h2"))
h1.start(); h2.start()
h1.join(); h2.join()
print("Con orden fijo + timeout no hay bloqueo permanente.")

## 3. Multiprocessing basico

Un **proceso** es un programa en ejecucion con su **propia memoria**. Como no comparten memoria, los procesos Python **no sufren el GIL** y pueden ejecutar codigo CPU-bound en paralelo real en multiples nucleos del CPU.

### 3.1 Crear procesos con `Process()`

API: `Process(target=func, args=...)`, luego `.start()` y `.join()`.

In [ ]:
import multiprocessing as mp

def calcular_cuadrado(n: int) -> None:
    print(f"Proceso {mp.current_process().name}: {n}^2 = {n ** 2}")

procesos = []
for numero in [2, 3, 4, 5]:
    p = mp.Process(target=calcular_cuadrado, args=(numero,))
    procesos.append(p)
    p.start()

for p in procesos:
    p.join()

print("Todos los procesos terminaron.")

### 3.2 Comunicacion con `Queue`

Como los procesos tienen memoria aislada, las variables globales no se comparten. Para comunicarse se usan **colas** (`Queue`) que serializan objetos via pickle.

**Importante**: en Python moderno (3.14+ con `spawn` por defecto), el codigo que lanza procesos debe estar dentro del bloque `if __name__ == "__main__":`, porque los procesos hijos vuelven a importar el modulo.

In [ ]:
def worker_nombre(cola: mp.Queue, nombre: str) -> None:
    cola.put(f"Hola, soy {nombre} (pid {mp.current_process().pid})")

if __name__ == "__main__":
    cola = mp.Queue()
    procs = [mp.Process(target=worker_nombre, args=(cola, f"P{i}")) for i in range(3)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()
    while not cola.empty():
        print(cola.get())

### 3.3 Comunicacion con `Pipe`

Un `Pipe` es un canal bidireccional entre **dos** procesos (o uno solo en cada extremo). Uno escribe (`send`), el otro lee (`recv`), y viceversa.

In [ ]:
def productor_pipe(conn) -> None:
    for dato in range(1, 6):
        conn.send(dato)
    conn.close()

if __name__ == "__main__":
    extremo_a, extremo_b = mp.Pipe()
    p = mp.Process(target=productor_pipe, args=(extremo_b,))
    p.start()
    p.join()
    while extremo_a.poll():
        print("Leido del pipe:", extremo_a.recv())

### 3.4 `spawn` vs `fork` (Python 3.14+)

Al crear un proceso, el sistema operativo debe copiar el programa. Hay **dos metodos de inicio** (start method):

- **`fork`**: el hijo es una **copia exacta** del padre en el momento de la creacion. Rapido, pero hereda todo (estado, locks, fds) y **no es seguro con hilos**. Era el default en Linux/macOS.
- **`spawn`**: el hijo **arranca un interprete nuevo** y vuelve a importar el modulo `__main__`. Mas lento (arranque completo), pero **mas seguro y portable**. Es el **default desde Python 3.14** en todas las plataformas.

> Consecuencia practica: con `spawn` (default en 3.14), el codigo que lanza procesos DEBE estar dentro de `if __name__ == "__main__":`.

```
        fork                              spawn
  padre -----> copia exacta         padre -----> nuevo interprete
  (rapido, hereda todo)                \
  X inseguro con hilos                 -> importa __main__ de nuevo
                                        (lento pero seguro y portable)
```

In [ ]:
import sys
print("Start method por defecto:", mp.get_start_method())
print("Metodos disponibles:", mp.get_all_start_methods())
print("Python:", sys.version.split()[0])

## 4. ProcessPoolExecutor

`concurrent.futures.ProcessPoolExecutor` es la forma moderna y limpia de paralelizar **tareas CPU-bound**. Gestiona un grupo de procesos reutilizables y ofrece la misma API que `ThreadPoolExecutor` (`submit`, `map`, `as_completed`).

### 4.1 `map` y `submit`

La funcion debe ser **picklable** (definida en el modulo, no lambda) porque se serializa hacia los procesos hijos.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def es_primo(n: int) -> bool:
    if n < 2:
        return False
    for d in range(2, int(n**0.5) + 1):
        if n % d == 0:
            return False
    return True

if __name__ == "__main__":
    numeros = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

    # map: aplica a toda la lista, devuelve en orden
    with ProcessPoolExecutor(max_workers=4) as pool:
        resultados = list(pool.map(es_primo, numeros))
    for n, r in zip(numeros, resultados):
        print(f"{n}: primo={r}")

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

if __name__ == "__main__":
    numeros = [104729, 104723, 104701, 100003, 999983, 982451653]
    with ProcessPoolExecutor(max_workers=4) as pool:
        futuros = {pool.submit(es_primo, n): n for n in numeros}
        for fut in as_completed(futuros):
            n = futuros[fut]
            print(f"{n}: primo={fut.result()}")
    print("as_completed devuelve en el orden en que terminan, no en el enviado.")

### 4.2 Speedup real en tarea CPU-bound

Vamos a calcular una suma grande y comparar secuencial vs procesos. Al ser CPU-bound (gasta nucleos), los **procesos** deben dar un speedup cercano al numero de nucleos.

In [ ]:
import multiprocessing

def suma_larga(limite: int, inicio_cuadrante: int) -> int:
    total = 0
    for i in range(inicio_cuadrante, inicio_cuadrante + limite):
        total += i ** 2
    return total

def secuencial_total(n, npar) -> int:
    chunk = n // npar
    return sum(suma_larga(chunk, i * chunk) for i in range(npar))

if __name__ == "__main__":
    N = 4_000_000
    WORKERS = min(8, multiprocessing.cpu_count())

    t0 = time.perf_counter()
    r_sec = secuencial_total(N, WORKERS)
    t_sec = time.perf_counter() - t0

    chunk = N // WORKERS
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=WORKERS) as pool:
        partes = list(pool.map(suma_larga, [chunk] * WORKERS, [i * chunk for i in range(WORKERS)]))
    r_par = sum(partes)
    t_par = time.perf_counter() - t0

    print(f"Nucleos usados : {WORKERS}")
    print(f"Resultados iguales: {r_sec == r_par}")
    print(f"Secuencial : {t_sec:.3f}s")
    print(f"Procesos   : {t_par:.3f}s")
    print(f"Speedup    : {t_sec / t_par:.2f}x")

### 4.3 Aproximacion de pi con procesos

Metodo de Monte Carlo: lanzar puntos aleatorios en un cuadrado y contar cuantos caen dentro del circulo. Es CPU-bound y paralelizable perfectamente.

In [ ]:
import random

def monte_carlo_pi(puntos: int, semilla: int) -> int:
    rng = random.Random(semilla)
    dentro = 0
    for _ in range(puntos):
        x = rng.random()
        y = rng.random()
        if x * x + y * y <= 1.0:
            dentro += 1
    return dentro

if __name__ == "__main__":
    TOTAL = 4_000_000
    WORKERS = min(4, multiprocessing.cpu_count())
    POR_PROC = TOTAL // WORKERS

    with ProcessPoolExecutor(max_workers=WORKERS) as pool:
        futuros = [pool.submit(monte_carlo_pi, POR_PROC, s) for s in range(WORKERS)]
        dentro_total = sum(f.result() for f in futuros)

    pi_aprox = 4 * dentro_total / TOTAL
    print(f"Pi aproximado: {pi_aprox:.6f}")
    print(f"Pi real      : {__import__('math').pi:.6f}")

## 5. Memoria compartida

Pasar datos entre procesos se hace normalmente con **pickle** (serializar -> enviar -> deserializar). Para estructuras grandes, eso es costoso. La **memoria compartida** permite a varios procesos leer/escribir la **misma region de memoria** sin copiar, lo que reduce drasticamente la transferencia.

### 5.1 `multiprocessing.Array` y `Value` con lock

`Value` y `Array` crean objetos en memoria compartida y pueden llevar lock para proteger la escritura.

In [ ]:
def acumular_en_shared(valor: mp.Value, veces: int) -> None:
    for _ in range(veces):
        with valor.get_lock():   # protege la escritura
            valor.value += 1

if __name__ == "__main__":
    contador = mp.Value("i", 0, lock=True)   # 'i' = entero con signo
    procs = [mp.Process(target=acumular_en_shared, args=(contador, 100_000))
             for _ in range(4)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()
    print(f"Valor compartido final (esperado 400000): {contador.value}")

In [ ]:
def escribir_en_array(arr, idx, valor) -> None:
    arr[idx] = valor  # 'd' = double con lock

if __name__ == "__main__":
    arr = mp.Array("d", 4, lock=True)   # array de 4 doubles
    procs = [mp.Process(target=escribir_en_array, args=(arr, i, i * 10.5))
             for i in range(4)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()
    print("Array compartido:", list(arr))

### 5.2 `multiprocessing.shared_memory.SharedMemory`

El modulo `shared_memory` (Python 3.8+) crea bloques de memoria con nombre que cualquier proceso puede abrir por nombre. Es la forma mas directa de compartir memoria. Idealmente se combina con `numpy.frombuffer` para numeros de alto rendimiento.

In [ ]:
from multiprocessing.shared_memory import SharedMemory

def llenar(shm_nombre: str) -> None:
    shm = SharedMemory(name=shm_nombre)
    buf = shm.buf
    for i in range(10):
        buf[i] = i * 2        # escribir bytes directamente
    shm.close()

if __name__ == "__main__":
    shm = SharedMemory(create=True, size=10)
    p = mp.Process(target=llenar, args=(shm.name,))
    p.start(); p.join()
    print("Contenido memoria compartida:", list(shm.buf))
    shm.close()
    shm.unlink()  # liberar el segmento

### 5.3 Memoria compartida vs pickle

| Aspecto | Pickle (cola/envio) | Memoria compartida |
|---|---|---|
| Mecanismo | Serializar -> copiar -> deserializar | Leer/escribir la misma RAM |
| Uso de memoria | Copias adicionales por envio | Un solo chunk, sin copias |
| Velocidad | Lenta para datos grandes | Muy rapida |
| Complejidad | Sencilla y segura | Requiere gestionar lock/liberacion |
| Caso ideal | Datos pequenos, muchos procesos | Matrices/arreglos grandes (numpy) |

> **Regla**: para voluminosos datos numericos, usa `shared_memory` o `Array`/`Value`. Para objetos pequenos e irregulares, pickle es suficiente.

## 6. Serializacion con pickle

Para pasar una funcion (y sus argumentos) a un proceso con `spawn` o `ProcessPoolExecutor`, Python la **serializa con pickle**: lo convierte a una secuencia de bytes que se envia y reconstruye en el hijo.

### 6.1 Que es pickle

`pickle` serializa objetos Python arbitrarios (no solo primitivos): listas, dicts, clases de instancias, funciones top-level, etc.

In [ ]:
import pickle

datos = {"usuario": "ana", "roles": ["admin", "editor"], "activo": True}
bytes_pk = pickle.dumps(datos)
print("Bytes serializados:", bytes_pk)
print("Original == Recuperado?", pickle.loads(bytes_pk) == datos)

# Incluso funciones top-level se serializan
def doble(x):
    return x * 2

f = pickle.loads(pickle.dumps(doble))
print("Funcion pickled llamada ->", f(21))

### 6.2 Que NO se puede picklear

No se pueden serializar:

- **Lambdas** (anónimas).
- **Clases o funciones definidas localmente** (dentro de otra funcion o dentro de `__main__` en un `if __name__`).
- Locks, sockets, generadores, algunos recursos del sistema.

Por eso, las funciones que pasas a `ProcessPoolExecutor` deben estar **definidas en el modulo**, en el nivel superior. Las lambdas fallan con `AttributeError` o `PicklingError`.

In [ ]:
def funcion_top_level(x):
    return x + 100

if __name__ == "__main__":
    # Funcion top-level: funciona
    with ProcessPoolExecutor(max_workers=2) as pool:
        print("Top-level:", list(pool.map(funcion_top_level, [1, 2, 3])))

    # Lambda NO se puede picklear -> dos hilos/procesos fallarian
    try:
        with ProcessPoolExecutor(max_workers=2) as pool:
            list(pool.map(lambda x: x + 100, [1, 2, 3]))
    except Exception as e:
        print("Lambda fallo con:", type(e).__name__, "->", str(e)[:60])

### 6.3 Guardas para procesos workers

Cuando Python crea procesos (sobre todo con `spawn`), el hijo **vuelve a importar el modulo** `__main__`. Si el codigo se ejecutara al importar, cada hijo lanzaria mas procesos -> recursion infinita / errores.

La solucion es la **guarda** `if __name__ == "__main__":`, que solo ejecuta el codigo en el proceso principal.

In [ ]:
# Patron correcto en scripts con procesos

if __name__ == "__main__":
    # Todo el codigo orientado a procesos va aqui.
    print("Este bloque solo corre en el proceso principal,")
    print("los workers lo importan pero no lo ejecutan.")

### 6.4 `cloudpickle` (intro)

`cloudpickle` es una extension de `pickle` que **si** serializa lambdas, closures y clases/funciones locales. La biblioteca **Dask** y otros frameworks de computacion distribuida la usan de base.

> Para instalarla: `pip install cloudpickle`. En entornos como Dask ya viene incluida.

No es parte de la libreria estandar, por eso aqui solo se demuestra la idea sin depender de ella. Su ventaja: flexibilidad para enviar codigo dinamico a workers.

## 7. Comparacion y benchmark

Vamos a resolver **el mismo problema** (I/O y CPU) de tres formas: secuencial, con hilos y con procesos. Asi vemos empiricamente cuando conviene cada uno.

In [ ]:
from concurrent.futures import ThreadPoolExecutor as TPE
from concurrent.futures import ProcessPoolExecutor as PPE

def tarea_cpu(t: float) -> float:
    # CPU-bound puro: calcular sin liberar el GIL
    acc = 0.0
    while acc < t:
        acc += 0.000001
    return acc

def tarea_io(t: float) -> float:
    # I/O-bound: esperar (libera el GIL)
    time.sleep(t)
    return t


In [ ]:
def correr(tipo, tiempo, trabajos, func):
    t0 = time.perf_counter()
    if tipo == "secuencial":
        for _ in range(trabajos):
            func(tiempo)
    elif tipo == "hilos":
        with TPE(max_workers=trabajos) as pool:
            list(pool.map(func, [tiempo] * trabajos))
    else:  # procesos
        with PPE(max_workers=trabajos) as pool:
            list(pool.map(func, [tiempo] * trabajos))
    return time.perf_counter() - t0


In [ ]:
if __name__ == "__main__":
    TRABAJOS = 8

    print("--- Problema CPU-bound (tiempo de trabajo ~0.15 CPU) ---")
    cpu_sec = correr("secuencial", 0.15, TRABAJOS, tarea_cpu)
    cpu_hil = correr("hilos", 0.15, TRABAJOS, tarea_cpu)
    cpu_pro = correr("procesos", 0.15, TRABAJOS, tarea_cpu)

    print("--- Problema I/O-bound (espera 0.2s) ---")
    io_sec = correr("secuencial", 0.2, TRABAJOS, tarea_io)
    io_hil = correr("hilos", 0.2, TRABAJOS, tarea_io)
    io_pro = correr("procesos", 0.2, TRABAJOS, tarea_io)

In [ ]:
if __name__ == "__main__":
    print("\nTabla de resultados (segundos, menor = mejor):")
    print("=" * 44)
    print(f"{'Estrategia':<12} {'CPU-bound':>10} {'I/O-bound':>10}")
    print("=" * 44)
    print(f"{'Secuencial':<12} {cpu_sec:>10.3f} {io_sec:>10.3f}")
    print(f"{'Hilos':<12} {cpu_hil:>10.3f} {io_hil:>10.3f}")
    print(f"{'Procesos':<12} {cpu_pro:>10.3f} {io_pro:>10.3f}")
    print("=" * 44)
    print()
    print("Conclusiones esperadas:")
    print("  1. CPU-bound: procesos >> secuencial, hilos NO ayudan (GIL).")
    print("  2. I/O-bound : hilos ~ procesos, ambos >> secuencial.")

### Cuando usar cada uno

| Situacion | Recomendacion |
|---|---|
| Muchas llamadas a API/DB (I/O) | `ThreadPoolExecutor` (hilos) |
| Leer/escribir muchos archivos (I/O) | Hilos o `asyncio` |
| Calculo numerico intensivo (CPU) | `ProcessPoolExecutor` (procesos) |
| Preprocesar imagenes/matrices grandes | Procesos + memoria compartida |
| Datos que deben compartirse en caliente | Memoria compartida (+ lock) |
| Simplicidad, datos pequenos | Secuencial o pickle |

```
   I/O-bound?  ---- si ---->  HILOS (saltan el GIL al esperar)
       |
       no
       v
   CPU-bound?  ---- si ---->  PROCESOS (paralelismo real)
                   |
                   no
                   v
              Secuencial simple (presupuesto bajo, pocos datos)
```

## Tabla de referencia rapida

### `concurrent.futures`

| Clase / Funcion | Descripcion |
|---|---|
| `ThreadPoolExecutor` | Grupo de hilos; ideal para I/O |
| `ProcessPoolExecutor` | Grupo de procesos; ideal para CPU |
| `executor.submit(fn, *args)` | Envia una tarea, devuelve `Future` |
| `executor.map(fn, *iterables)` | Aplica fn en paralelo, devuelve en orden |
| `as_completed(futures)` | Iterador de futuros en orden de finalizacion |
| `Future.result(timeout)` | Resultado (bloquea si falta, timeout opcional) |
| `Future.done()` / `cancel()` | Estado del futuro |

### `multiprocessing`

| Clase / Funcion | Descripcion |
|---|---|
| `Process(target, args)` | Crear un proceso |
| `p.start()` / `p.join()` | Iniciar y esperar el proceso |
| `Queue` | Cola FIFO para comunicar procesos |
| `Pipe` | Canal bidireccional entre dos extremos |
| `Value(type, valor, lock)` | Variable escalar en memoria compartida |
| `Array(type, n, lock)` | Arreglo primitivo en memoria compartida |
| `shared_memory.SharedMemory` | Bloque de memoria con nombre (sin pickle) |
| `cpu_count()` | Numero de nucleos del CPU |
| `get_start_method()` | Metodo de inicio actual (fork/spawn) |

### `threading`

| Clase / Funcion | Descripcion |
|---|---|
| `Thread(target, args, daemon)` | Crear un hilo |
| `t.start()` / `t.join(timeout)` | Iniciar y esperar |
| `t.is_alive()` | Esta vivo? |
| `Lock` | Exclusion mutua |
| `RLock` | Lock reentrante |
| `Event` | Señal esperar/avisar |
| `Condition` | Coordinacion productor/consumidor |

## Ejercicios

### Ejercicio 1 (guiado): Descarga paralela de URLs

Completa el codigo para descargar varias URLs en paralelo con `ThreadPoolExecutor`. La funcion `simular_descarga` duerme `delay` segundos y devuelve el contenido.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def simular_descarga(url: str, delay: float = 0.3) -> str:
    time.sleep(delay)
    return f"contenido_de_{url}"

urls = ["a.com", "b.com", "c.com", "d.com"]

# TODO: usa ThreadPoolExecutor con max_workers=4 y map() para
# descargar todas las urls en paralelo y guardar los resultados.
with ThreadPoolExecutor(max_workers=4) as pool:
    resultados = list(pool.map(simular_descarga, urls))

for r in resultados:
    print(r)

### Ejercicio 2 (guiado): Race condition y Lock

Corrige la race condition del contador usando un `threading.Lock`. Tu solucion debe llegar siempre a `50000`.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

contador = 0
lock = threading.Lock()  # TODO: usa este lock para proteger la suma

def sumar(veces: int) -> None:
    global contador
    for _ in range(veces):
        with lock:              # protege la seccion critica
            contador += 1

with ThreadPoolExecutor(max_workers=5) as pool:
    list(pool.map(sumar, [10000] * 5))

print(f"Contador final (a menudo != 50000 sin lock): {contador}")

### Ejercicio 3 (guiado): Factorizacion paralela con procesos

Completa el codigo para factorizar varios numeros en paralelo con `ProcessPoolExecutor`. Recuerda la guarda `if __name__ == "__main__":` y que la funcion debe ser top-level.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def factorizar(n: int) -> list[int]:
    factores = []
    d = 2
    while n > 1:
        while n % d == 0:
            factores.append(d)
            n //= d
        d += 1
    return factores

if __name__ == "__main__":
    numeros = [360, 97, 1024, 123456]
    with ProcessPoolExecutor(max_workers=4) as pool:
        resultado = list(pool.map(factorizar, numeros))
    for n, fac in zip(numeros, resultado):
        print(f"{n} = {fac}")

### Ejercicio independiente: Procesador paralelo de numeros

Construye un sistema que procese una lista grande de numeros con `ProcessPoolExecutor` y utilice **memoria compartida** (`mp.Array`) para acumular resultados agregados.

Requisitos:
- Divide un rango de numeros en `WORKERS` trozos.
- Cada proceso calcula la suma de sus trozos y la escribe en una posicion del `mp.Array`.
- El proceso principal lee el array compartido y suma para obtener el resultado total.
- Muestra tambien el speedup frente a la version secuencial.

**Pista**: define una funcion top-level que reciba `(inicio, fin, arr)` y escriba `arr[indice] = suma(inicio..fin)` usando `arr.get_lock()`.

In [ ]:
import multiprocessing as mp
import time

def sumar_rango(inicio: int, fin: int, arr) -> None:
    total = sum(range(inicio, fin))
    indice = arr_is_active  # placeholder, sobreescribir
    with arr.get_lock():
        arr[0] = total

if __name__ == "__main__":
    # --- Completa aqui la solucion ---
    # 1. Define WORKERS, TAMANO y un mp.Array('i', ...)
    # 2. Reparte [0, TAMANO) entre los procesos en paralelo
    # 3. Suma los resultados del array compartido
    # 4. Compara con la version secuencial
    print("Ejercicio independiente: implementa tu solucion aqui.")

## Resumen

- **Hilos** (`threading`, `ThreadPoolExecutor`) comparten memoria, son baratos y excelentes para **I/O**, pero el **GIL** impide el paralelismo real en **CPU-bound**.
- **Procesos** (`multiprocessing`, `ProcessPoolExecutor`) tienen memoria aislada, superan el GIL y dan **paralelismo real en CPU**, a cambio de mayor costo y serializacion con pickle.
- La **sincronizacion** (`Lock`, `RLock`, `Event`, `Condition`) protege datos compartidos y evita race conditions y deadlocks (orden fijo + timeouts).
- La **comunicacion** entre procesos usa `Queue`, `Pipe`, `Value`/`Array` o `shared_memory`; elegir memoria compartida para datos grandes.
- **pickle** serializa funciones y datos top-level, pero **no lambdas ni definiciones locales**; `cloudpickle` amplia ese limite.
- Con **`spawn`** (default en Python 3.14), usa siempre `if __name__ == "__main__":` y funciones top-level.

**Decision final**:
```
   I/O-bound  ->  HILOS   ->  ThreadPoolExecutor
   CPU-bound  ->  PROCESOS -> ProcessPoolExecutor
   Ambos      ->  combinar, o asyncio para I/O + procesos para CPU
```